In [9]:
import os
import re
import gc

import pandas as pd
import matplotlib.pyplot as plt

from pyulog import ULog

In [10]:
# return the array of combined log file line
def getLogData(baseLogDir, dateTime, iteration, testCase, model="iris"):
	logPath = os.path.join(baseLogDir, dateTime, iteration, model, testCase, "log-combined.log_plain.log")
	log = None

	if os.path.exists(logPath):
		with open(logPath, "r") as f:
			log = f.readlines()

	return log

In [11]:
# return the ulg file name parsed from the combined log
def findUlgName(log):
	pattern = r"INFO\s+\[logger\]\s+Opened full log file:\s+(.*\.ulg)"
	ulg_file_name = ""

	match = re.search(pattern, log)

	if match:
		ulg_file_name = match.group(1)

	return ulg_file_name

In [12]:
# return the ulog parsed from the ulg file
def getUlogData(baseUlgDir, ulgFileName):
	ulog = None

	normPath = os.path.normpath(ulgFileName)
	ulgPath = os.path.join(baseUlgDir, normPath)

	if os.path.exists(ulgPath):
		ulog = ULog(ulgPath)

	return ulog

In [13]:
# ulg, combined log default location
baseLogDir = os.path.expanduser("~/ws/PX4-Autopilot/logs")
baseUlgDir = os.path.expanduser("~/ws/PX4-Autopilot/build/px4_sitl_default/tmp_mavsdk_tests/rootfs")

# combined log path data
experimentDateTime = "2025-03-21T21-42-37Z"
testCase = "normal_hold_20.f"

# get the combined log data
combinedLog = getLogData(baseLogDir, experimentDateTime, "001", testCase)

# parse the ulg file name from the combined log
ulgFileName = findUlgName("".join(combinedLog))

# get the ulog data
ulog = getUlogData(baseUlgDir, ulgFileName)


# get the dataset from the ulog
'''
vehicle_angular_velocity_groundtruth =  pd.DataFrame(ulog.get_dataset("vehicle_attitude_groundtruth").data)
vehicle_attitude_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_attitude_groundtruth").data)
vehicle_local_position_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_local_position_groundtruth").data)
vehicle_global_position_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_global_position_groundtruth").data)

print("vehicle_angular_velocity_groundtruth \n", vehicle_angular_velocity_groundtruth.head(1))
print("vehicle_attitude_groundtruth \n", vehicle_attitude_groundtruth.head(1))
print("vehicle_local_position_groundtruth \n", vehicle_local_position_groundtruth.head(1))
print("vehicle_global_position_groundtruth \n", vehicle_global_position_groundtruth.head(1))
'''

innov_df = pd.DataFrame(ulog.get_dataset("estimator_innovation_variances").data)
# 4) actuator_armed에서 Arm 시점(timestamp) 추출
armed_df = pd.DataFrame(ulog.get_dataset("actuator_armed").data)
arm_events = armed_df[armed_df["armed"] == 1]
arm_time = arm_events["timestamp"].iloc[0] if not arm_events.empty else None

# 5) Arm 이후부터 로그 마지막까지 gps_hpos 통계 계산
if arm_time is not None:
	flight_df = innov_df[innov_df["timestamp"] >= arm_time]
else:
	flight_df = innov_df

gps_Xseries = flight_df["gps_hpos[0]"]
gps_Yseries = flight_df["gps_hpos[0]"]
mean_Xgps = gps_Xseries.mean()
mean_Ygps = gps_Xseries.mean()
max_Xgps  = gps_Xseries.max()
max_Ygps  = gps_Yseries.max()
min_Xgps  = gps_Xseries.min()
min_Ygps  = gps_Yseries.min()

# 메모리 정리
del ulog, innov_df, armed_df, flight_df
gc.collect()

info = { "gps_hpos[0]": {
		"mean": mean_Xgps,
		"max": max_Xgps,
		"min": min_Xgps
	},
	"gps_hpos[1]": {
		"mean": mean_Ygps,
		"max": max_Ygps,
		"min": min_Ygps
	}}
print(info)

{'gps_hpos[0]': {'mean': 1.1263928, 'max': 1.1284143, 'min': 1.1255032}, 'gps_hpos[1]': {'mean': 1.1263928, 'max': 1.1284143, 'min': 1.1255032}}


In [6]:
vehicle_angular_velocity_groundtruth['timestamp'] = pd.to_datetime(vehicle_angular_velocity_groundtruth['timestamp'], unit='us')
vehicle_angular_velocity_groundtruth.set_index("timestamp", inplace=True)
interpolated = vehicle_angular_velocity_groundtruth.interpolate(method="time")
interpolated

,timestamp_sample,q[0],q[1],q[2],q[3],delta_q_reset[0],delta_q_reset[1],delta_q_reset[2],delta_q_reset[3],quat_reset_counter
timestamp,,,,,,,,,,
1970-01-01 00:00:18.496,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.500,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.508,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.516,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.524,0,0.707110,0.000393,0.000394,0.707110,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:02:25.636,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
1970-01-01 00:02:25.648,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
1970-01-01 00:02:25.656,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
